**This section is dedicated to functions that compute geometrical elements, used in the simulation engine to compute physics, or in certain other algorithms such as heading correction, or potential field obstacle avoidance.**



**Problem 1:** A robot is pointing at an angle $\theta$. How can we wrap this angle back to the $[-\pi, \pi)$ interval so that it knows how to reorient itself?

So basically we're identifying a mapping:
$$\text{wrap}: \mathbb{R} \rightarrow [-\pi, \pi)$$

Because angles in a circle is cyclic mod $2\pi$, we can take the modulo: 
$$ \theta _0 \equiv \theta \mod 2 \pi$$
This gives us the mapping $\mathbb{R} \rightarrow [0,2\pi)$. For our desired range for the function, we can do:
$$\text{wrap} (\theta) = \theta - 2 \pi \left\lfloor \dfrac{\theta + \pi}{2\pi}  \right\rfloor$$
And because $x \mod m = x - m \lfloor \frac{x}{m} \rfloor$, this can be implemented easily in code.

**Problem 2: For a rectangular robot, suppose we know its pose in the world coordinates and its dimensions, how do we compute the coordinates of its vertices in the world frame?**

In the robot frame, if we let the origin to be the center of the rectangle, then define the half length along the $x$ and $y$ axes as $h_x$, $h_y$ respectively, the vertices in local coordinates are:
$$
\mathbf{v}_1 = \begin{bmatrix}h_x \\ h_y \end{bmatrix} \hspace{40pt}
\mathbf{v}_2 = \begin{bmatrix}h_x \\ -h_y \end{bmatrix} \hspace{40pt}
\mathbf{v}_3 = \begin{bmatrix}-h_x \\ -h_y \end{bmatrix} \hspace{40pt}
\mathbf{v}_4 = \begin{bmatrix}-h_x \\ h_y \end{bmatrix}
$$
If we also define the position of the robot in the world coordinate as $\mathbf{p} = \begin{bmatrix}x \\ y \end{bmatrix}$, we can add the rotation matrix about the origin with the position to give us the final four vertices. In matrix form:
$$
\mathbf{v}_i^W = R(\theta)\mathbf{v}_i^L + \mathbf{p}
$$
where $R(\theta)$ is the rotation matrix:
$$ R(\theta) = \begin{bmatrix} \cos \theta & -\sin \theta \\ \sin \theta & \cos \theta \end{bmatrix}

**Problem 3:** For a potential field, we want to find the shortest distance from our robot to a static object, aligned with the world's axes. How do we determine the shortest distance from an OBB (Oriented Bounding Box), our robot, to these AABBs (Axis-Aligned Bounding Box)?

First we can approximate this by finding the shortest distance from a point to an AABB. For ease of computation, we're going to use the AABB's local coordinates to get symmetry. For a point $P_W(x,y)$ in the world coordinates, in this frame it would be $P_L(x-c_x, y-c_y)$, where $C(c_x,c_y)$ is the AABB's center. Again, define the half length along the axes as $h_x$ and $h_y$, we can find the point on the AABB that is closest to $P$. Let that point be $Q$. We can compute $Q$ as:
$$Q(\text{clamp}(x_{P_L}, -h_x, h_x), \text{clamp}(y_{P_L}, -h_y, h_y))$$
So finally, our minimum distance is just:
$$ d(P, AABB) = \sqrt{(x_{P_L} - x_Q)^2 + (y_{P_L} - y_Q)^2}$$
Let $\mathbf{v} = \overrightarrow{QP_L}$. The normal vector pointing from the point on the box to our point $P$ is exactly:
$$\mathbf{n} = \dfrac{\mathbf{v}}{||\mathbf{v}||}$$
Knowing the distance and the normal vector is crucial for potential field calculations.

But this might fail in certain cases. Our robot is not a sphere. A point will not correctly encapsulate the geometry of the robot. For potential field to be precise and smooth even when our rectangular robot is turning, we will need general OBB to AABB.

To find $d(OBB, AABB)$, we would want to first prove a lemma.

**Lemma 1:** For two convex polygons, the closest pair of points must involve at least one vertex.

*Proof:* Parametrise the two convex polygons as $A(s)$ and $B(t)$. We define the squared distance function:
$$f(s,t) = ||A(s) - B(t)||^2$$
We are just choosing this to avoid square roots in our computations. The minimiser remains the same because the map $x \mapsto x^2$ is strictly increasing for nonnegative real numbers.

We know that for any mapping $\mathbb{R}^n \mapsto \mathbb{R}^n$ on a closed region, the extrema only exists at the boundary points, or in interior points satisfying:
$$\nabla f = 0$$
Now take a closer look at the parametrised polygons. They are piecewise linear. Except for the vertex, at every point, the derivative of the parametrised arc is a constant. The vertices are the boundaries of these piecewise linear intervals. If the extrema are there, then we already know that they already involve at least a vertex. 

We then consider the case were they are inside the lines. Then:
$$\dfrac{\partial f}{\partial s} = 0 \hspace{20pt} \text{and} \hspace{20pt} \dfrac{\partial f}{\partial t} = 0$$
Using the chain rule, we yield:
$$
2(A-B)\frac{\partial A}{\partial s} = 0\\
-2(A-B)\frac{\partial B}{\partial t} = 0
$$
This means that the tangent line with $A$ at that point is perpendicular to $A-B$, which is just the vector $\overrightarrow{BA}$. Similarly, we have $\overrightarrow{t_B} \perp \overrightarrow{BA}$. This means that $t_A || t_B$, and in turn means that their exist a pair of edges that are parallel to each other, each belonging to a different polygon. With this, we can "slide" the line segment whose length is the shortest distance along an edge, eventually meeting an endpoint, which means it also involves a vertex.

Now that we know this, we can build the algorithm to find $d(OBB, AABB)$. Assuming the OBB and the AABB do not overlap, we can separate the algorithm into four parts:

1. Transform the coordinates to AABB frame
2. Check the distances of the 4 vertices of the OBB to the AABB
3. Transform the coordinates to OBB frame
4. Check the distances of the 4 vertices of the AABB to the OBB

Now, let the OBB have center $C_O (c_x, c_y)$, orientation $\theta$, and half extents $(h_x, h_y)$. Let the AABB have center $C_A (a_x, a_y)$ and half extents $(w_x, w_y)$.

First, we transform the OBB vertices coordinates from the world frame to the AABB frame:
$$ V_{OBB}^A = V_{OBB}^W - C_A$$
Then, we calculate the distances:
$$d(V_{OBB}^A, AABB)$$
Then we transfrom the AABB vertices to the OBB frame:
$$ V_{AABB}^O = R^T(\theta) V_{AABB}^W + C_O$$
And we calculate the distances:
$$d(V_{AABB}^O, OBB)$$
Finally we yield 8 points in a set $D$, and our distance is the smallest element in that set.

**Remark:** The same algorithm still holds for general distance between a point and a randomly oriented convex polygon, or two randomly oriented convex polygons. 

**Problem 4:** A robot is going head first into the long part of a shelf. We need to calculate a tangent term so it can follow along the wall until it's free. How do we compute and choose which tangent to use?

Let's say we already know the pair of points that satisfy the shortest distance between the obstacle and the robot. That, alongside with the robot's heading vector $\mathbf{h}$, we can compute this directly. We know:
$$\mathbf{n} = \begin{bmatrix} \mathbf{n_x} \\ \mathbf{n_y} \end{bmatrix} \Rightarrow 
\mathbf{t} = \begin{bmatrix} -\mathbf{n_y} \\ \mathbf{n_x} \end{bmatrix} $$
But remember that $-\mathbf{t}$ is also a valid solution. Then, we simply compute the dot product $\mathbf{t.h}$. If it's positive, then $\cos \theta < 90 ^\circ$, and vice versa. We choose the positive solution to minimise turning cost.